# Primero recopilamos


In [1]:
from pathlib import Path
import pandas as pd


In [2]:

carpeta = Path(".")

archivo_catalogo = carpeta / "formulario_limpio_1.xlsx"
sheet_catalogo = "Catálogo VRAI"

archivos_output = list(carpeta.glob("output_*.xlsx")) + list(carpeta.glob("output_single_*.xlsx"))
archivos_output = [f for f in archivos_output if f.name != archivo_catalogo.name]

# Catálogo VRAI: mapeo por NRC
df_catalogo = pd.read_excel(archivo_catalogo, sheet_name=sheet_catalogo, dtype=str)

columnas_catalogo = [
    "NRC",
    "Sigla",
    "Horario Cátedra/Clase",
    "Horario Ayudantía",
    "Horario Laboratorio",
    "Horario Taller",
    "Horario Terreno",
]

mapa_catalogo = (
    df_catalogo[columnas_catalogo]
    .dropna(subset=["NRC"])
    .assign(NRC=lambda x: x["NRC"].astype(str).str.strip())
    .drop_duplicates(subset=["NRC"], keep="first")
)

# Archivos output: filas a recopilar
columnas_output = [
    "RUT UC",
    "Nombre",
    "Email",
    "NRC",
    "Unidad Académica",
]

dfs = []

for archivo in archivos_output:
    df = pd.read_excel(archivo, dtype=str, sheet_name="Con éxito")

    df = df[columnas_output].copy()
    df["NRC"] = df["NRC"].astype(str).str.strip()

    dfs.append(df)

df_final = pd.concat(dfs, ignore_index=True)

# Cruce por NRC
df_final = df_final.merge(mapa_catalogo, on="NRC", how="left")

# Orden final
df_final = df_final[
    [
        "RUT UC",
        "Nombre",
        "Email",
        "NRC",
        "Unidad Académica",
        "Sigla",
        "Horario Cátedra/Clase",
        "Horario Ayudantía",
        "Horario Laboratorio",
        "Horario Taller",
        "Horario Terreno",
    ]
]

In [3]:
df_final

,RUT UC,Nombre,Email,NRC,Unidad Académica,Sigla,Horario Cátedra/Clase,Horario Ayudantía,Horario Laboratorio,Horario Taller,Horario Terreno
0,118606K,Elaine Victoria Garcia,elaine.garcia@tufts.edu,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
1,1185799,Mathilde Marie Blandine Lagarde,mathilde.lagrd@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
2,118539K,Vadim MAURER-DECOUT,vadim.maurerdecout@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
3,1186299,Ava Louise Giere,a.giere@wustl.edu,10218,Letras,LET1005,CLAS = L:1100 a 1210; W:1100 a 1210;,NaN,NaN,TAL = V:0940 a 1050;,NaN
4,1184148,Clara Fornés Roig,fornesroigc@gmail.com,10261,Estetica,EST210A,CLAS = M:1450 a 1720;,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
595,1184598,Julian Staeva-Vieira,julian.staeva_vieira@tufts.edu,29824,Agronomia,AGC209,CLAS = L:1100 a 1330;,NaN,NaN,NaN,NaN
596,1186892,Joana Haristoy,joana.haristoy@opendeusto.es,30100,Escuela de Medicina,MED855,CLAS = J:1100 a 1330;,NaN,NaN,NaN,NaN
597,1184520,Leslie Santiago Andres,lc23-0707@lclark.edu,30374,Ciencia Politica,ICP0720,CLAS = L:1220 a 1330; W:1220 a 1330;,NaN,NaN,NaN,NaN
598,1185748,Ximena Herrera Monreal,A01705214@itesm.mx,33476,Derecho,DNO158,CLAS = V:0820 a 1050;,NaN,NaN,NaN,NaN


In [4]:
# === AGREGAR CARGA DESDE EXCEL DE EXTRANJEROS + EXCEL DE NRC PROGRAMADOS ===

archivo_extranjeros = carpeta / "VRAIEcono.xls"
archivo_nrc_extra = carpeta / "DetalleProgramados.xlsx"

# columnas a completar por ti
col_rut = "rut"
col_dv = "dig_verif"
col_nombre = "apellido_paterno"
col_email = "direccion_internet"
col_nrc_extranjeros = "crn"
col_unidad = "nom_unidad_academ"

col_nrc_nrc_extra = "NRC"
col_sigla_nrc_extra = "Materia"
col_sigla_nrc_extra_verif = "Número Curso"
col_horario_catedra_nrc_extra = "Cátedra"
col_horario_ayudantia_nrc_extra = "Ayudantía"
col_horario_laboratorio_nrc_extra = "Laboratorio"
col_horario_taller_nrc_extra = "Taller"
col_horario_terreno_nrc_extra = "Terreno"

# leer extranjeros
df_extranjeros_raw = pd.read_excel(archivo_extranjeros, dtype=str)

df_extranjeros_raw = df_extranjeros_raw[
    (df_extranjeros_raw["ano_proceso_extrj"].astype(str).str.strip() == "2026") &
    (df_extranjeros_raw["cod_periodo_proceso_extrj"].astype(str).str.strip() == "21")
].copy()

df_extranjeros = pd.DataFrame({
    "RUT UC": (
        df_extranjeros_raw[col_rut].astype(str).str.strip() +
        df_extranjeros_raw[col_dv].astype(str).str.strip()
    ),
    "Nombre": df_extranjeros_raw[col_nombre],
    "Email": df_extranjeros_raw[col_email],
    "NRC": df_extranjeros_raw[col_nrc_extranjeros].astype(str).str.strip(),
    "Unidad Académica": df_extranjeros_raw[col_unidad]
})

# leer nrc programados
df_nrc_extra_raw = pd.read_excel(archivo_nrc_extra, dtype=str)


WARNING *** file size (1339377) not 512 + multiple of sector size (512)
WARNING *** OLE2 inconsistency: SSCS size is 0 but SSAT size is non-zero


In [5]:

mapa_nrc_extra = pd.DataFrame({
    "NRC": df_nrc_extra_raw[col_nrc_nrc_extra].astype(str).str.strip(),
    "Sigla": df_nrc_extra_raw[col_sigla_nrc_extra] +
    df_nrc_extra_raw[col_sigla_nrc_extra_verif].astype(str).str.strip(),
    "Horario Cátedra/Clase": df_nrc_extra_raw[col_horario_catedra_nrc_extra],
    "Horario Ayudantía": df_nrc_extra_raw[col_horario_ayudantia_nrc_extra],
    "Horario Laboratorio": df_nrc_extra_raw[col_horario_laboratorio_nrc_extra],
    "Horario Taller": df_nrc_extra_raw[col_horario_taller_nrc_extra],
    "Horario Terreno": df_nrc_extra_raw[col_horario_terreno_nrc_extra],
}).dropna(subset=["NRC"]).drop_duplicates(subset=["NRC"], keep="first")

# cruzar y anexar al df_final ya existente
df_extranjeros = df_extranjeros.merge(mapa_nrc_extra, on="NRC", how="left")

df_extranjeros = df_extranjeros[
    [
        "RUT UC",
        "Nombre",
        "Email",
        "NRC",
        "Unidad Académica",
        "Sigla",
        "Horario Cátedra/Clase",
        "Horario Ayudantía",
        "Horario Laboratorio",
        "Horario Taller",
        "Horario Terreno",
    ]
]


In [6]:

df_final = pd.concat([df_final, df_extranjeros], ignore_index=True)

In [7]:
df_final

,RUT UC,Nombre,Email,NRC,Unidad Académica,Sigla,Horario Cátedra/Clase,Horario Ayudantía,Horario Laboratorio,Horario Taller,Horario Terreno
0,118606K,Elaine Victoria Garcia,elaine.garcia@tufts.edu,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
1,1185799,Mathilde Marie Blandine Lagarde,mathilde.lagrd@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
2,118539K,Vadim MAURER-DECOUT,vadim.maurerdecout@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN
3,1186299,Ava Louise Giere,a.giere@wustl.edu,10218,Letras,LET1005,CLAS = L:1100 a 1210; W:1100 a 1210;,NaN,NaN,TAL = V:0940 a 1050;,NaN
4,1184148,Clara Fornés Roig,fornesroigc@gmail.com,10261,Estetica,EST210A,CLAS = M:1450 a 1720;,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...
1779,289507892,LEVI,colombe.levi@sciencespo.fr,17336,Historia,NaN,NaN,NaN,NaN,NaN,NaN
1780,289507892,LEVI,colombe.levi@sciencespo.fr,29468,Comunicaciones,NaN,NaN,NaN,NaN,NaN,NaN
1781,289510362,KUMAR,tejasakumar@gmail.com,33127,Economía y Administración,EAA209L,CLAS = M:1100 a 1210; J:1100 a 1210;,NaN,NaN,NaN,NaN
1782,289539069,ZHOU,jeslynzhou@u.nus.edu,33127,Economía y Administración,EAA209L,CLAS = M:1100 a 1210; J:1100 a 1210;,NaN,NaN,NaN,NaN


In [8]:

# Guardar resultado
df_final.to_excel("recopilado_outputs.xlsx", index=False)

In [9]:
df_final = pd.read_excel("recopilado_outputs.xlsx")

# Ahora chequeamos

In [10]:
modulos = {
    1: "08:20",
    2: "09:40",
    3: "11:00",
    4: "12:20",
    5: "13:30",
    6: "14:50",
    7: "17:30",
    8: "18:50",
    9: "20:10"
}

In [11]:
archivo_eval = "Material Docente 24-02-2026_FACEA.xlsx"

df_eval = pd.read_excel(
    archivo_eval,
    skiprows=11,
    header=[0, 1]
)


In [12]:
df_eval

Unnamed: 0_level_0 Unnamed: 1_level_0 Unnamed: 2_level_0  \
    Unnamed: 0_level_1              Sigla               Sec.   
0                  NaN            EAA1110                  1   
1                  NaN            EAA1110                  2   
2                  NaN            EAA1110                  3   
3                  NaN            EAA1110                  4   
4                  NaN            EAA1110                  5   
..                 ...                ...                ...   
382                NaN            EAG190A                  1   
383                NaN            EAG165E                  1   
384                NaN            EAG160E                  1   
385                NaN                NaN                NaN   
386                NaN                NaN                NaN   

    Unnamed: 3_level_0                                 Unnamed: 4_level_0  \
                  Créd                                   Nombre del curso   
0                   10                      COMPORTAMIENTO ORGANIZACIONAL   
1                   10                      COMPORTAMIENTO ORGANIZACIONAL   
2                   10                      COMPORTAMIENTO ORGANIZACIONAL   
3                   10                      COMPORTAMIENTO ORGANIZACIONAL   
4                   10                      COMPORTAMIENTO ORGANIZACIONAL   
..                 ...                                                ...   
382                 10  EXPLORANDO EL MUNDO DEL TRABAJO PARA LA TOMA D...   
383                 10  POLÍTICAS PÚBLICAS: DERRIBANDO MITOS DE LA POB...   
384                 10                          ¿CÓMO TOMAMOS DECISIONES?   
385                NaN                                                NaN   
386                NaN                                                NaN   

          Unnamed: 5_level_0                    Unnamed: 6_level_0  Clases  \
                        Prof                            Requisitos Horario   
0                   G. MUÑOZ                              ADMISIÓN   M-J:2   
1              S. VALENZUELA                              ADMISIÓN   M-J:4   
2              M.E. VIGNEAUX                              ADMISIÓN   M-J:4   
3                  E. KAUSEL                              ADMISIÓN   L-W:3   
4              S. VALENZUELA                              ADMISIÓN   M-J:5   
..                       ...                                   ...     ...   
382                  K. LUND                     CIENCIAS SOCIALES   M-J:5   
383  G. UGARTE / F. BASTIDAS                     CIENCIAS SOCIALES   L-W:2   
384                 C. MUÑOZ                     CIENCIAS SOCIALES   M-J:6   
385                      NaN                                   NaN     NaN   
386                      NaN  CURSOS DICTADOS POR OTRAS FACULTADES     NaN   

    Ayudantía                        Pruebas               \
      Horario Horario.1           Día/Módulo Día/Módulo.1   
0         V:6       NaN  29/abr:7 y 29/may:7          NaN   
1         V:6       NaN  29/abr:7 y 29/may:7          NaN   
2         V:6       NaN  29/abr:7 y 29/may:7          NaN   
3         V:6       NaN  29/abr:7 y 29/may:7          NaN   
4         V:6       NaN  29/abr:7 y 29/may:7          NaN   
..        ...       ...                  ...          ...   
382       V:5       NaN                   PF          NaN   
383       V:2       NaN             24/abr:2          NaN   
384         -       NaN   9/abr:6 y 14/may:6          NaN   
385       NaN       NaN                  NaN          NaN   
386       NaN       NaN                  NaN          NaN   

                  Examen                        Prueba Recuperativa            
                     Día           Hora Hora.1                  Día      Hora  
0    2024-07-03 00:00:00   8:20 - 10:20    NaN  2024-07-03 00:00:00  17:30:00  
1    2024-07-03 00:00:00   8:20 - 10:20    NaN  2024-07-03 00:00:00  17:30:00  
2    2024-07-03 00:00:00   8:20 - 10:20 

In [13]:

df_eval = df_eval[[("Unnamed: 1_level_0", "Sigla"),
                   ("Pruebas", "Día/Módulo"),
                   ("Examen", "Día"),
                   ("Examen", "Hora")]]

df_eval.columns = ["Sigla", "Pruebas", "Examen Dia", "Examen Hora"]
df_eval = df_eval.dropna(subset=["Sigla"]).reset_index(drop=True)
df_eval = df_eval.drop_duplicates(subset="Sigla").reset_index(drop=True)

In [14]:
df_eval

,Sigla,Pruebas,Examen Dia,Examen Hora
0,EAA1110,29/abr:7 y 29/may:7,2024-07-03 00:00:00,8:20 - 10:20
1,EAA1210,13/abr:7 y 26/may:7,2025-07-09 00:00:00,8:20 - 10:20
2,EAA1220,13/abr:7 y 26/may:7,2025-07-10 00:00:00,8:20 - 10:20
3,EAA2230,13/abr:7 y 26/may:7,2024-06-30 00:00:00,13:30 - 15:30
4,EAA220B,14/abr:7 y 1/jun:7,2024-07-09 00:00:00,13:30 - 15:30
...,...,...,...,...
169,CURSOS DE FORMACION GENERAL DICTADOS POR LA FA...,NaN,NaN,NaN
170,EAG170A,17/abr:2,2025-07-01 00:00:00,13:30 - 15:30
171,EAG190A,PF,PF,PF
172,EAG165E,24/abr:2,2025-07-01 00:00:00,13:30 - 15:30


In [15]:
siglas_relevantes = set(df_final["Sigla"].unique())

df_eval = df_eval[df_eval["Sigla"].isin(siglas_relevantes)].reset_index(drop=True)

In [16]:
df_eval

,Sigla,Pruebas,Examen Dia,Examen Hora
0,EAA1110,29/abr:7 y 29/may:7,2024-07-03 00:00:00,8:20 - 10:20
1,EAA1210,13/abr:7 y 26/may:7,2025-07-09 00:00:00,8:20 - 10:20
2,EAA1220,13/abr:7 y 26/may:7,2025-07-10 00:00:00,8:20 - 10:20
3,EAA2220,10/abr:7 y 15/may:7,2024-07-02 00:00:00,8:20 - 10:20
4,EAA2310,29/abr:7 y 29/may:7,2024-07-04 00:00:00,8:20 - 10:20
5,EAA2320,17/abr:7 y 29/may:7,2024-07-08 00:00:00,8:20 - 10:20
6,EAA2410,13/abr:7 y 26/may:7,2024-07-02 00:00:00,8:20 - 10:20
7,EAA2110,30/abr:7 y 11/jun:7,2024-07-03 00:00:00,8:20 - 10:20
8,EAA2420,23/abr:7 y 10/jun:7,2024-07-06 00:00:00,13:30 - 15:30
9,EAA1510,27/abr:7 y 5/jun:7,2024-07-08 00:00:00,13:30 - 15:30


In [17]:

# Año a usar para las pruebas
anio_pruebas = 2026

meses = {
    "ene": 1, "feb": 2, "mar": 3, "abr": 4, "may": 5, "jun": 6,
    "jul": 7, "ago": 8, "sep": 9, "oct": 10, "nov": 11, "dic": 12
}

dias_semana = {
    0: "L",
    1: "M",
    2: "W",
    3: "J",
    4: "V",
    5: "S",
    6: "D"
}

import re

def extraer_fechas_prueba(texto):
    
    if pd.isna(texto):
        return []

    texto = str(texto).strip()
    if texto == "PF":
        return []

    fechas = []

    # busca patrones tipo 13/abr:7, 8/abr:5-6, 8/abr:5; 13/abr:5, etc.
    coincidencias = re.findall(r"(\d{1,2})/([a-zA-Z]+)\s*:\s*([\d\-]+)", texto)

    for dia_txt, mes_txt, modulo_txt in coincidencias:
        dia = int(dia_txt)
        mes = meses[mes_txt.strip().lower()]
        fecha = pd.Timestamp(year=anio_pruebas, month=mes, day=dia)

        if "-" in modulo_txt:
            modulo_inicio = int(modulo_txt.split("-")[0])
        else:
            modulo_inicio = int(modulo_txt)

        fechas.append({
            "fecha": fecha,
            "dia_semana": dias_semana[fecha.weekday()],
            "modulo": modulo_inicio
        })

    return fechas


In [18]:


df_eval["Pruebas parseadas"] = df_eval["Pruebas"].apply(extraer_fechas_prueba)



In [19]:
df_eval["Pruebas parseadas"][20]

[{'fecha': Timestamp('2026-04-23 00:00:00'), 'dia_semana': 'J', 'modulo': 7},
 {'fecha': Timestamp('2026-06-10 00:00:00'), 'dia_semana': 'W', 'modulo': 7}]

In [20]:
import re
from datetime import datetime, timedelta

columnas_horario = [
    "Horario Cátedra/Clase",
    "Horario Ayudantía",
    "Horario Laboratorio",
    "Horario Taller",
    "Horario Terreno",
]

def hhmm_a_min(hhmm):
    hhmm = str(hhmm).replace(":", "").strip()
    return int(hhmm[:2]) * 60 + int(hhmm[2:])

def bloques_desde_horario(texto):
    if pd.isna(texto):
        return []
    
    texto = str(texto).strip()
    if texto == "":
        return []

    coincidencias = re.findall(r'([LMWJV]):\s*(\d{3,4})\s*a\s*(\d{3,4})', texto)
    
    bloques = []
    for dia, inicio, fin in coincidencias:
        bloques.append({
            "dia_semana": dia,
            "inicio_min": hhmm_a_min(inicio),
            "fin_min": hhmm_a_min(fin)
        })
    return bloques


In [21]:

# bloques ocupados por fila/curso
df_final["bloques_ocupados"] = df_final[columnas_horario].apply(
    lambda fila: sum((bloques_desde_horario(fila[col]) for col in columnas_horario), []),
    axis=1
)


In [22]:
df_final

,RUT UC,Nombre,Email,NRC,Unidad Académica,Sigla,Horario Cátedra/Clase,Horario Ayudantía,Horario Laboratorio,Horario Taller,Horario Terreno,bloques_ocupados
0,118606K,Elaine Victoria Garcia,elaine.garcia@tufts.edu,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 740, 'fin_m..."
1,1185799,Mathilde Marie Blandine Lagarde,mathilde.lagrd@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 740, 'fin_m..."
2,118539K,Vadim MAURER-DECOUT,vadim.maurerdecout@gmail.com,10169,Agronomia,AGC220,CLAS = M:1220 a 1720;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 740, 'fin_m..."
3,1186299,Ava Louise Giere,a.giere@wustl.edu,10218,Letras,LET1005,CLAS = L:1100 a 1210; W:1100 a 1210;,NaN,NaN,TAL = V:0940 a 1050;,NaN,"[{'dia_semana': 'L', 'inicio_min': 660, 'fin_m..."
4,1184148,Clara Fornés Roig,fornesroigc@gmail.com,10261,Estetica,EST210A,CLAS = M:1450 a 1720;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 890, 'fin_m..."
...,...,...,...,...,...,...,...,...,...,...,...,...
1779,289507892,LEVI,colombe.levi@sciencespo.fr,17336,Historia,NaN,NaN,NaN,NaN,NaN,NaN,[]
1780,289507892,LEVI,colombe.levi@sciencespo.fr,29468,Comunicaciones,NaN,NaN,NaN,NaN,NaN,NaN,[]
1781,289510362,KUMAR,tejasakumar@gmail.com,33127,Economía y Administración,EAA209L,CLAS = M:1100 a 1210; J:1100 a 1210;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 660, 'fin_m..."
1782,289539069,ZHOU,jeslynzhou@u.nus.edu,33127,Economía y Administración,EAA209L,CLAS = M:1100 a 1210; J:1100 a 1210;,NaN,NaN,NaN,NaN,"[{'dia_semana': 'M', 'inicio_min': 660, 'fin_m..."


In [23]:
df_final["bloques_ocupados"][0]

[{'dia_semana': 'M', 'inicio_min': 740, 'fin_min': 1040}]

In [24]:
df_final[df_final["Nombre"] == "Paul Eisenblätter"]

,RUT UC,Nombre,Email,NRC,Unidad Académica,Sigla,Horario Cátedra/Clase,Horario Ayudantía,Horario Laboratorio,Horario Taller,Horario Terreno,bloques_ocupados
210,1162969,Paul Eisenblätter,eisenblaetter.paul@gmail.com,27961,Economia y Administracion,EAA3501,CLAS = W:1450 a 1720;,AYU = V:1220 a 1330;,NaN,NaN,NaN,"[{'dia_semana': 'W', 'inicio_min': 890, 'fin_m..."
274,1162969,Paul Eisenblätter,eisenblaetter.paul@gmail.com,29585,Economia y Administracion,EAA3401,CLAS = L:1220 a 1330; W:1220 a 1330;,AYU = V:0820 a 0930;,NaN,NaN,NaN,"[{'dia_semana': 'L', 'inicio_min': 740, 'fin_m..."


In [25]:

# mapa de evaluaciones por sigla
mapa_pruebas = df_eval.set_index("Sigla")["Pruebas parseadas"].to_dict()


In [26]:
mapa_pruebas["EAA3401"]

[{'fecha': Timestamp('2026-05-28 00:00:00'), 'dia_semana': 'J', 'modulo': 7}]

In [27]:

def bloque_evaluacion(prueba):
    inicio = datetime.strptime(modulos[prueba["modulo"]], "%H:%M")
    fin = inicio + timedelta(hours=2)
    return {
        "dia_semana": prueba["dia_semana"],
        "inicio_min": inicio.hour * 60 + inicio.minute,
        "fin_min": fin.hour * 60 + fin.minute,
        "fecha": prueba["fecha"]
    }

def hay_tope(b1, b2):
    return (
        b1["dia_semana"] == b2["dia_semana"]
        and b1["inicio_min"] < b2["fin_min"]
        and b2["inicio_min"] < b1["fin_min"]
    )


In [28]:
def min_a_hora(minutos):
    horas = minutos // 60
    mins = minutos % 60
    return f"{horas:02d}:{mins:02d}"

topes = []

for rut, grupo in df_final.groupby("RUT UC"):
    cursos = grupo.to_dict("records")
    
    for curso_eval in cursos:
        sigla_eval = curso_eval["Sigla"]
        pruebas = mapa_pruebas.get(sigla_eval, [])
        
        if not pruebas:
            continue
        
        for prueba in pruebas:
            bloque_prueba = bloque_evaluacion(prueba)
            
            for otro_curso in cursos:
                if otro_curso["Sigla"] == sigla_eval:
                    continue
                
                for bloque_ocupado in otro_curso["bloques_ocupados"]:
                    if hay_tope(bloque_prueba, bloque_ocupado):
                        topes.append({
                            "RUT UC": rut,
                            "Nombre": curso_eval["Nombre"],
                            "Email": curso_eval["Email"],
                            "Sigla evaluacion": sigla_eval,
                            "NRC evaluacion": curso_eval["NRC"],
                            "Fecha evaluacion": bloque_prueba["fecha"],
                            "Dia evaluacion": bloque_prueba["dia_semana"],
                            "Inicio evaluacion": min_a_hora(bloque_prueba["inicio_min"]),
                            "Fin evaluacion": min_a_hora(bloque_prueba["fin_min"]),
                            "Sigla curso tope": otro_curso["Sigla"],
                            "NRC curso tope": otro_curso["NRC"],
                            "Bloque curso tope": f'{bloque_ocupado["dia_semana"]}:{bloque_ocupado["inicio_min"]}-{bloque_ocupado["fin_min"]}',
                            "Unidad Académica": curso_eval["Unidad Académica"],
                        })


In [29]:

df_topes = pd.DataFrame(topes).drop_duplicates().reset_index(drop=True)


In [30]:
df_topes

,RUT UC,Nombre,Email,Sigla evaluacion,NRC evaluacion,Fecha evaluacion,Dia evaluacion,Inicio evaluacion,Fin evaluacion,Sigla curso tope,NRC curso tope,Bloque curso tope,Unidad Académica
0,1164422,JAKUBONYTE,nikolete.jakubonyte@edu.uah.es,EAG160E,33888,2026-04-09,J,14:50,16:50,EAG190A,27260,J:890-960,Economía y Administración
1,1164422,JAKUBONYTE,nikolete.jakubonyte@edu.uah.es,EAG160E,33888,2026-05-14,J,14:50,16:50,EAG190A,27260,J:890-960,Economía y Administración
2,1185756,Giulianna Yetselli Rodriguez Pintos,giuliannarodriguez22@gmail.com,EAA1510,27948,2026-04-27,L,17:30,19:30,EAA2260,28083,L:1050-1200,Economia y Administracion
3,118587K,CIRILLO,francesco.cirillo@studbocconi.it,EAG165E,29714,2026-04-24,V,09:40,11:40,EAA2220,27881,V:580-650,Economía y Administración
4,1185942,BOUXIN,melchior.bouxin@student-cs.fr,EAA1510,27948,2026-04-27,L,17:30,19:30,EAA1220,25966,L:1050-1120,Economía y Administración
5,118654K,Raphaël Louis Gérard Bourrat,rbourrat05@gmail.com,EAA2420,27946,2026-06-10,W,17:30,19:30,EAA361A,19289,W:1050-1120,Economia y Administracion
6,118654K,Raphaël Louis Gérard Bourrat,rbourrat05@gmail.com,EAE2510,28069,2026-06-03,W,17:30,19:30,EAA361A,19289,W:1050-1120,Economia y Administracion
7,118654K,BOURRAT,rbourrat05@gmail.com,EAE2510,33215,2026-06-03,W,17:30,19:30,EAA361A,19289,W:1050-1120,Economía y Administración
8,118654K,BOURRAT,rbourrat05@gmail.com,EAA2420,27946,2026-06-10,W,17:30,19:30,EAA361A,19289,W:1050-1120,Economía y Administración
9,1187228,DIMARTINO,emma.dimartino@studbocconi.it,EAA1110,27870,2026-04-29,W,17:30,19:30,EAE1220,24705,W:1050-1120,Economía y Administración


In [31]:

df_topes.to_excel("topes_evaluaciones.xlsx", index=False)